In [1]:
from langchain_community.document_loaders.json_loader import JSONLoader
from pathlib import Path
from pprint import pp

/var/folders/c8/p_3zr7pj43g93mjqt5_f7vfc0000gn/T/ipykernel_4237/3260479200.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.json_loader import JSONLoader


In [2]:
## define file path

file_path = Path("../knowledge-source/apparels.json")
# check if file is correct and exists
print(file_path.exists())

True


In [4]:
# metadata_func: A custom callback used by JSONLoader to enrich the metadata of each loaded Document.
#
# Parameters:
#   data     (dict) : A single JSON record extracted by jq_schema (e.g., one product object).
#                     Use this to pull domain-specific fields you want stored as metadata.
#   metadata (dict) : The base metadata dict pre-populated by JSONLoader with two default keys:
#                       'source'  -> absolute file path of the JSON file
#                       'seq_num' -> 1-based index of the record within the file
#                     Mutate and return this dict to customise what gets attached to each Document.
#
# Returns:
#   dict : The updated metadata dict that is attached to the LangChain Document object.

def metadata_func(data: dict, metadata: dict) -> dict:
    # Assign product-specific fields from the raw JSON record into the metadata dict.
    metadata['productID'] = data['productID']       # Unique product identifier
    metadata['category'] = data['category']         # Product category (e.g., 'Men Cloths', 'Bags')
    metadata['manufacturer'] = data['manufacturer'] # Brand / manufacturer name (e.g., 'Zara')

    # To strip an auto-added key from metadata, use del — e.g., to remove the record index:
    # del metadata['seq_num']

    return metadata

In [5]:
loader = JSONLoader(
    file_path=file_path.as_posix(),  # Absolute path to the JSON file; .as_posix() ensures cross-platform path format
    jq_schema='.products[]',         # jq filter: selects every element inside the top-level 'products' array — yields one Document per product
    content_key="Description",       # JSON field whose value becomes Document.page_content (the text that will be embedded & retrieved)
    metadata_func=metadata_func      # Custom callback (data, metadata) -> dict to attach extra fields (productID, category, manufacturer) to Document.metadata
)

In [6]:
# load documents
document = loader.load()

In [7]:
# print metadata
print(document)

[Document(metadata={'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 1, 'productID': '0000001', 'category': 'Men Cloths', 'manufacturer': 'Zara'}, page_content='Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.'), Document(metadata={'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 2, 'productID': '0000002', 'category': 'Men Cloths', 'manufacturer': 'Zara'}, page_content='Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.'), Document(metadata={'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 3, 'productID': '0000003', 'category': 'Perfumes', 'manufacturer': 'Zara'}, page_content='Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.'), Doc

In [8]:
# print page_content present in each document
for doc in document:
    print(doc.page_content, end="\n\n")

Oversize-fit coat made of a viscose blend fabric. Notch lapel collar and long sleeves with buttoned cuffs.

Trench coat made of technical fabric with a velvety finish. Notch lapel collar and long sleeves.

Parka made of technical fabric, padded on inside. High neck with a hood and long sleeves with elasticated cuffs.

The first Vibrant Leather for le benjamin with its fruity hint of pineapple that gives it modernity, accompanied by cedar notes, adding a woody touch of rejuvenation.

Multicoloured backpack. Soft construction. Featuring one main zip pocket and a medium-sized zip pocket on the inside.



In [9]:
# print metadata
for doc in document:
    print(doc.metadata,end="\n \n")

{'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 1, 'productID': '0000001', 'category': 'Men Cloths', 'manufacturer': 'Zara'}
 
{'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 2, 'productID': '0000002', 'category': 'Men Cloths', 'manufacturer': 'Zara'}
 
{'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 3, 'productID': '0000003', 'category': 'Perfumes', 'manufacturer': 'Zara'}
 
{'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 4, 'productID': '0000004', 'category': 'Perfumes', 'manufacturer': 'Zara'}
 
{'source': '/Users/anujsingh/learning/advnace-rag/RAG-DOCUMENT-LOADER/knowledge-source/apparels.json', 'seq_num': 5, 'productID': '0000005', 'category': 'Bags', 'manufacturer': 'Zara'}
 


In [10]:
# print length of documents
print(f"total documents : {len(document)}")

total documents : 5
